# 第4章 Zigbee
O-QPSKの復調、DSSSの逆拡散、FCSの検査を経て、MACとNWKのヘッダを読みます。最後にコラムのThreadも試します。

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, "..")  # code/ 直下のモジュールを読み込む
DATA = Path(os.environ.get("SDR_BOOK_DATA", "../data"))  # IQ データの置き場所
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import zigbee as zb

x = zb.load_iq(DATA / "zigbee_ch20.iq")
bursts = zb.detect_bursts(x)
print("bursts:", len(bursts))

## 参照パターンとSHRの相関

In [ ]:
scores = []
for s, e in bursts:
    score, signs, pos = zb.find_shr(x[max(0, s - 400):e + 400])
    scores.append(score)
plt.figure(figsize=(9, 3)); plt.plot(scores, "."); plt.axhline(0.6 * len(zb.SHR), ls="--")
plt.xlabel("burst"); plt.ylabel("SHR score"); plt.show()

## 逆拡散 → FCS → MAC / NWK

In [ ]:
for (s, e), score in zip(bursts, scores):
    if score < 0.6 * len(zb.SHR):
        continue
    _, signs, pos = zb.find_shr(x[max(0, s - 400):e + 400])
    psdu = zb.read_ppdu(signs, pos)
    if not zb.check_fcs(psdu):
        continue
    mac = zb.parse_mac(psdu)
    line = f"{s / zb.SAMPLE_RATE:7.3f}s type={mac['frame_type']} seq={mac['seq']} len={len(psdu)}"
    if mac["frame_type"] == 1 and mac["dst_mode"] == 2 and len(mac["payload"]) >= 8:
        nwk = zb.parse_nwk(mac["payload"])
        line += f" 0x{mac['src_addr']:04X}->0x{mac['dst_addr']:04X} NWK type={nwk['frame_type']} counter={nwk.get('frame_counter')}"
    print(line)

## コラム：Thread
同じ物理層のコードでThreadのフレームを読みます（`thread.py`）。

In [ ]:
import thread

path = DATA / "thread_ch11.iq"
if not path.exists():
    path = Path("thread_test.iq")
    thread.make_test_iq(path)
thread.main(path)